# Training Relationformer su Kaggle

Notebook per clonare il progetto, installare le dipendenze compatibili con Kaggle, validare l'intero dataset configurato e avviare il training multi-GPU.

Prima di eseguirlo:
- abilita un acceleratore GPU nelle impostazioni del notebook;
- collega il dataset `pid2graph-patched`;
- abilita Internet per clonare il repository e scaricare i pesi ImageNet di ResNet-101.

## 1. Configurazione dell'ambiente Kaggle

La cella seguente clona il branch di training in `/kaggle/working`, imposta i seed e verifica CUDA. Modifica `DATASET_ROOT_OVERRIDE` solo se il rilevamento automatico non trova il dataset.

In [ ]:
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/SimoneSgalla053/test-training-p3id.git"
BRANCH = "test/training"
WORKDIR = Path("/kaggle/working/relationformer")
DATASET_ROOT_OVERRIDE = None  # Esempio: "/kaggle/input/pid2graph-patched/Patched"
SEED = 10

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

if (WORKDIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "github", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only", "github", BRANCH], check=True)
else:
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORKDIR)],
        check=True,
    )

os.chdir(WORKDIR)
print("Repository:", WORKDIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU disponibili:", torch.cuda.device_count())
for gpu_index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_index)
    print(f"  GPU {gpu_index}: {props.name}, {props.total_memory / 1024**3:.1f} GiB")

assert torch.cuda.is_available(), "Abilita un acceleratore GPU nelle impostazioni Kaggle."

## 2. Installazione dei requirements

Viene mantenuta la versione di PyTorch/CUDA già installata da Kaggle. Se Internet è disabilitato, la cella accetta le dipendenze già presenti soltanto se gli import richiesti riescono.

In [ ]:
requirements = WORKDIR / "requirements-kaggle.txt"
assert requirements.is_file(), f"File mancante: {requirements}. Esegui il push del branch aggiornato."

install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
    text=True,
)
if install.returncode != 0:
    print("pip non ha completato l'installazione; verifico se i pacchetti sono già disponibili...")

import ignite
import monai
import scipy
import tensorboardX
import yaml

print("Dipendenze disponibili:")
print("  MONAI", monai.__version__)
print("  Ignite", ignite.__version__)
print("  SciPy", scipy.__version__)

## 3. Import e configurazione del training

I parametri principali vengono letti dalla configurazione versionata. Il notebook verifica i valori essenziali prima di utilizzare GPU e tempo di esecuzione.

In [ ]:
CONFIG_PATH = WORKDIR / "configs" / "road_2D.yaml"
with CONFIG_PATH.open(encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

assert config["MODEL"]["NUM_CLASSES"] == 11
assert config["MODEL"]["NUM_EDGE_CLASSES"] == 3
assert config["MODEL"]["DECODER"]["OBJ_TOKEN"] + config["MODEL"]["DECODER"]["RLN_TOKEN"] == 401
assert config["TRAIN"]["EPOCHS"] == 80
assert config["DATA"]["VALIDATION_PERCENT"] == 5

print("Configurazione training:")
print("  batch/GPU:", config["DATA"]["BATCH_SIZE"])
print("  batch globale:", config["DATA"]["BATCH_SIZE"] * torch.cuda.device_count())
print("  epoche:", config["TRAIN"]["EPOCHS"])
print("  learning rate:", config["TRAIN"]["LR"])
print("  train sources:", config["DATA"]["TRAIN_SOURCES"])
print("  test sources:", config["DATA"]["TEST_SOURCES"])
print("  limite sessione:", config["TRAIN"]["MAX_HOURS"], "ore")

## 4. Caricamento dell'intero dataset

Il rilevamento cerca la cartella che contiene direttamente le tre sorgenti richieste. Tutti i campioni validi vengono indicizzati; non viene applicato alcun sottocampionamento. OPEN100 resta nel test per evitare contaminazione.

In [ ]:
REQUIRED_SOURCES = {"Dataset PID", "PID2Graph Synthetic", "PID2Graph OPEN100"}

if DATASET_ROOT_OVERRIDE:
    dataset_root = Path(DATASET_ROOT_OVERRIDE)
    candidates = [dataset_root]
else:
    candidates = sorted(
        {
            marker.parent
            for marker in Path("/kaggle/input").rglob("Dataset PID")
            if marker.is_dir()
        }
    )

valid_roots = [
    candidate
    for candidate in candidates
    if REQUIRED_SOURCES.issubset({child.name for child in candidate.iterdir() if child.is_dir()})
]
assert len(valid_roots) == 1, (
    "Impossibile determinare un'unica root del dataset. "
    f"Candidate valide: {[str(path) for path in valid_roots]}. "
    "Imposta DATASET_ROOT_OVERRIDE nella prima cella."
)
dataset_root = valid_roots[0].resolve()

source_stats = {}
for source_name in sorted(REQUIRED_SOURCES):
    source_root = dataset_root / source_name
    graph_paths = list(source_root.rglob("*.graphml"))
    png_paths = list(source_root.rglob("*.png"))
    paired = sum(graph_path.with_suffix(".png").is_file() for graph_path in graph_paths)
    source_stats[source_name] = {
        "graphml": len(graph_paths),
        "png": len(png_paths),
        "paired": paired,
    }
    assert paired == len(graph_paths) == len(png_paths), (
        f"Coppie PNG/GraphML incomplete in {source_name}: {source_stats[source_name]}"
    )

os.environ["PID2GRAPH_DATA_PATH"] = str(dataset_root)
os.environ["RELATIONFORMER_CACHE_DIR"] = "/kaggle/working/relationformer-cache"
Path(os.environ["RELATIONFORMER_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

print("Dataset root:", dataset_root)
for source_name, stats in source_stats.items():
    print(f"  {source_name}: {stats}")
print("Totale coppie:", sum(stats["paired"] for stats in source_stats.values()))

## 5. Preprocessing e DataLoader

Questa fase costruisce una sola volta l'indice, ridimensiona le immagini a 512×512 in una memmap sotto `/kaggle/working` e valida le annotazioni. Il processo di training riutilizzerà la cache senza ricostruirla.

In [ ]:
preprocess_code = """
from train import load_config
from dataset_road_network import build_road_network_data

config = load_config('configs/road_2D.yaml', verbose=False)
train_dataset, validation_dataset = build_road_network_data(config, mode='split')
print(f'Train samples validi: {len(train_dataset)}')
print(f'Validation samples validi: {len(validation_dataset)}')
assert len(train_dataset) > 0 and len(validation_dataset) > 0
"""
subprocess.run([sys.executable, "-c", preprocess_code], cwd=WORKDIR, check=True)

## 6. Inizializzazione del modello

Il preflight seguente costruisce Relationformer con ResNet-101, matcher e loss multi-classe, li trasferisce sulla prima GPU e verifica il numero di parametri. Il training ricreerà questi oggetti nel processo distribuito.

In [ ]:
model_preflight_code = """
import torch
from train import load_config
from models import build_model
from models.matcher import build_matcher
from losses import SetCriterion

config = load_config('configs/road_2D.yaml', verbose=False)
device = torch.device('cuda:0')
model = build_model(config).to(device)
matcher = build_matcher(config)
criterion = SetCriterion(config, matcher, model)
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f'Modello inizializzato su {device}; parametri addestrabili: {trainable:,}')
del criterion, matcher, model
torch.cuda.empty_cache()
"""
subprocess.run([sys.executable, "-c", model_preflight_code], cwd=WORKDIR, check=True)

## 7. Esecuzione del training

Questa cella esegue tutte le epoche e tutti i batch configurati, usando automaticamente tutte le GPU visibili, AMP, validazione, logging, scheduler, early stopping e checkpoint del progetto. Il limite di 11 ore termina il training in modo pulito prima del limite Kaggle.

In [ ]:
gpu_ids = [str(index) for index in range(torch.cuda.device_count())]
assert gpu_ids, "Nessuna GPU disponibile."

training_command = [
    sys.executable,
    "train.py",
    "--config",
    "configs/road_2D.yaml",
    "--cuda_visible_device",
    *gpu_ids,
]
print("Avvio:", " ".join(training_command), flush=True)
subprocess.run(training_command, cwd=WORKDIR, check=True)

## 8. Salvataggio di checkpoint e metriche

Il trainer salva checkpoint, log testuali ed eventi TensorBoard. La cella crea inoltre un archivio scaricabile in `/kaggle/working`, senza includere la cache del dataset.

In [ ]:
import json

run_dir = WORKDIR / "trained_weights" / "runs" / "pid2graph_kaggle_10"
assert run_dir.is_dir(), f"Directory del run non trovata: {run_dir}"

checkpoints = sorted((run_dir / "models").glob("*.pt"), key=lambda path: path.stat().st_mtime)
print("Checkpoint prodotti:")
for checkpoint in checkpoints:
    print(f"  {checkpoint.name}: {checkpoint.stat().st_size / 1024**2:.1f} MiB")

manifest = {
    "repository_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "dataset_root": str(dataset_root),
    "dataset_stats": source_stats,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "checkpoints": [checkpoint.name for checkpoint in checkpoints],
}
with (run_dir / "kaggle_manifest.json").open("w", encoding="utf-8") as manifest_file:
    json.dump(manifest, manifest_file, indent=2)
shutil.copy2(CONFIG_PATH, run_dir / "road_2D.yaml")

archive_base = Path("/kaggle/working/relationformer_artifacts")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=run_dir)
print("Archivio pronto:", archive_path)

### Riprendere un training interrotto

Carica un checkpoint tra gli input Kaggle e imposta `RESUME_CHECKPOINT` al suo percorso. Lascia `None` per non eseguire questa cella.

In [ ]:
RESUME_CHECKPOINT = None  # Esempio: "/kaggle/input/mio-checkpoint/checkpoint_epoch=12.pt"

if RESUME_CHECKPOINT is None:
    print("Resume non richiesto.")
else:
    resume_path = Path(RESUME_CHECKPOINT)
    assert resume_path.is_file(), f"Checkpoint non trovato: {resume_path}"
    resume_command = [*training_command, "--resume", str(resume_path)]
    print("Ripresa:", " ".join(resume_command), flush=True)
    subprocess.run(resume_command, cwd=WORKDIR, check=True)